<a href="https://colab.research.google.com/github/GittyCyber22/BCO7006---1/blob/main/Code_Before.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
#main_code_for_subject_enrollement
catalogue={
    "BCO7006":{"name":"Coding for BA","prereqs":[],"semester":1,"female_only":0},
    "BCO7000":{"name":"Biz analytics","prereqs":["BCO7006"],"semester":1,"female_only":0},
    "BCO6008":{"name":"Predictive Analytics","prereqs":["BCO7000"],"semester":2,"female_only":0},
    "BCO7007":{"name":"Machine Learning","prereqs":["BCO7006","BCO7000"],"semester":2,"female_only":0},
    "WOM1000":{"name":"Women in STEM","prereqs":[],"semester":0,"female_only":1}
}

alice={"st_name":"Alice","st_id":"S001","gender":"Female","unit_passed":["BCO7000","BCO7006"]}
bahlil={"st_name":"Bahlil","st_id":"S002","gender":"Male","unit_passed":["BCO7000"]}
prabowo={"st_name":"Prabowo","st_id":"S003","gender":"Male","unit_passed":[]}
anas={"st_name":"Anis","st_id":"S004","gender":"Female","unit_passed":["BCO7006"]}
eric = {"st_name": "Eric", "st_id": "S005", "gender": "Male", "unit_passed": ["BCO7006"]}
erica = {"st_name": "Eric", "st_id": "S005", "gender": "Female", "unit_passed": ["BCO7006", "BCO7000"]}

In [3]:
#more advanced:

from dataclasses import dataclass, field
from typing import List
from IPython.core.display import clear_output


@dataclass
class Unit:
    code: str
    name: str
    prereqs: List[str]=field(default_factory=list)
    semester: int=1
    female_only: int=0 #if 0= open to all

@dataclass
class Student:
    st_name: str
    st_id: str
    gender: str
    unit_passed: List[str]=field(default_factory=list)
    semester: int=1


CATALOGUE=[
        Unit("BCO7006","Coding for BA",prereqs=[],semester=1,female_only=0),
        Unit("BCO7000","Biz analytics",prereqs=["BCO7006"],semester=1,female_only=0),
        Unit("BCO6008","Predictive Analytics",prereqs=["BCO7006"],semester=2,female_only=0),
        Unit("BCO7007","Machine Learning",prereqs=["BCO7000", "BCO7006"],semester=2,female_only=0),
        Unit("WOM1000","Women in STEM",prereqs=[],semester=0,female_only=1)
    ]

In [4]:
def can_enrol(student, unit_code,catalogue):
  """Return (ok,message) for single unit"""
  if unit_code not in catalogue:
    return (False,f"Unit {unit_code} not found")

  unit=catalogue[unit_code]

  #check prereq
  missing=[p for p in unit["prereqs"] if p not in student["unit_passed"]]
  # missing=[p for p in unit.prereqs if p not in student.unit_passed]
  if missing:
    return (False,f"Cannot enrol in {unit_code}. Missing prereqs {','.join(missing)}")

  #check conditions
  if unit["female_only"] and student["gender"].lower()!="female":
    return (False,f"Cannot enrol in {unit_code}. Only for female")
  return (True,f"Enrolment OK for {unit_code}{unit['name']}")

In [5]:
def process_request(student,requested_units,semester,catalogue,enrolments,next_code):
  """Process an enrolment reques, transform 'enrolments ' and return the next code"""
  print(f"Processing request for {student['st_name']}")

#cap checking = no more than 2
  if len(requested_units>2):
    print(f"Rejecting request for {student['st_name']}")
    return next_code

    for unit_code in requested_units:
      ok,message=can_enrol(student,unit_code,catalogue)
      if ok:
        record={
          "enrol_code":next_code,
          "unit_code":unit_code,
          "st_id":student["st_id"],
          "semester":semester
        }
        enrolments.append(record)
        next_code+=1
        print(f"OK:{message}->enrol_code:{next_code}")
      if not ok:
        print(f"Fail")
        return next_code

In [6]:
class EnrolmentSystemMin:
  """Min version of the screening system"""
  MAX_UNITS = 2 #

  def __init__(self, catalogue, enrolments=None, next_code=1001):
    self.catalogue = catalogue # Corrected 'set.catalogue' to 'self.catalogue'
    self.enrolments = enrolments if enrolments is not None else []
    self.next_code = next_code

  def check_student_eligibility(self, student, requested_units):
    """Checks if a student can enrol in a list of units, considering MAX_UNITS and individual unit eligibility.
       Returns a list of (ok, message) tuples for each requested unit.
    """
    results = []
    if len(requested_units) > self.MAX_UNITS:
      return [(False, f"Cannot enrol in more than {self.MAX_UNITS} units. Requested: {len(requested_units)}")]

    for unit_code in requested_units:
      ok, message = can_enrol(student, unit_code, self.catalogue) # Reusing
      results.append((ok, message))
    return results

  def enrol_student(self, student, requested_units, semester):
    """Attempts to enrol a student in the requested units.
       Returns True if all units are successfully enrolled, False otherwise.
    """
    print(f"Processing enrolment request for {student['st_name']}...")
    eligibility_results = self.check_student_eligibility(student, requested_units)

    if len(eligibility_results) == 1 and not eligibility_results[0][0]: # >2
      print(f"Rejected: {eligibility_results[0][1]}")
      return False

    all_ok = True
    for i, (ok, message) in enumerate(eligibility_results):
      unit_code = requested_units[i]
      if ok:
        record = {
            "enrol_code": self.next_code,
            "unit_code": unit_code,
            "st_id": student["st_id"],
            "semester": semester
        }
        self.enrolments.append(record)
        print(f"Enrolment OK for {unit_code}: {message} (Enrol Code: {self.next_code})")
        self.next_code += 1
      else:
        print(f"Enrolment FAILED for {unit_code}: {message}")
        all_ok = False
    return all_ok

##Out Put of every Scenario

1: Valid Enrolment
2:

In [7]:
# Initialize
enrolment_system = EnrolmentSystemMin(catalogue=catalogue)

print("Initial enrolments:", enrolment_system.enrolments)
print("Next enrolment code:", enrolment_system.next_code)

#  Test Code
print("\n--- Test Case 1: Valid enrolment (single unit) ---")
requested_units_1 = ["BCO6008", "BCO7007"]
success = enrolment_system.enrol_student(alice, requested_units_1, semester=1)
print(f"Enrolment status: {success}")
print("Current enrolments:", enrolment_system.enrolments)

print("\n--- Test Case 2 ---")
requested_units_1 = ["BCO7007"]
success = enrolment_system.enrol_student(bahlil, requested_units_1, semester=1)
print(f"Enrolment status: {success}")
print("Current enrolments:", enrolment_system.enrolments)

print("\n--- Test Case 3 ---")
requested_units_1 = ["BCO6008", "BCO7007"]
success = enrolment_system.enrol_student(prabowo, requested_units_1, semester=1)
print(f"Enrolment status: {success}")
print("Current enrolments:", enrolment_system.enrolments)

print("\n--- Test Case 4 ---")
requested_units_1 = ["WOM1000"]
success = enrolment_system.enrol_student(anas, requested_units_1, semester=1)
print(f"Enrolment status: {success}")
print("Current enrolments:", enrolment_system.enrolments)

print("\n--- Test Case 5 ---")
requested_units_1 = ["WOM1000"]
success = enrolment_system.enrol_student(eric, requested_units_1, semester=1)
print(f"Enrolment status: {success}")
print("Current enrolments:", enrolment_system.enrolments)

print("\n--- Test Case 6 ---")
requested_units_1 = ["BCO7006","BCO6008","WOM1000"]
success = enrolment_system.enrol_student(erica, requested_units_1, semester=1)
print(f"Enrolment status: {success}")
print("Current enrolments:", enrolment_system.enrolments)

Initial enrolments: []
Next enrolment code: 1001

--- Test Case 1: Valid enrolment (single unit) ---
Processing enrolment request for Alice...
Enrolment OK for BCO6008: Enrolment OK for BCO6008Predictive Analytics (Enrol Code: 1001)
Enrolment OK for BCO7007: Enrolment OK for BCO7007Machine Learning (Enrol Code: 1002)
Enrolment status: True
Current enrolments: [{'enrol_code': 1001, 'unit_code': 'BCO6008', 'st_id': 'S001', 'semester': 1}, {'enrol_code': 1002, 'unit_code': 'BCO7007', 'st_id': 'S001', 'semester': 1}]

--- Test Case 2 ---
Processing enrolment request for Bahlil...
Rejected: Cannot enrol in BCO7007. Missing prereqs BCO7006
Enrolment status: False
Current enrolments: [{'enrol_code': 1001, 'unit_code': 'BCO6008', 'st_id': 'S001', 'semester': 1}, {'enrol_code': 1002, 'unit_code': 'BCO7007', 'st_id': 'S001', 'semester': 1}]

--- Test Case 3 ---
Processing enrolment request for Prabowo...
Enrolment FAILED for BCO6008: Cannot enrol in BCO6008. Missing prereqs BCO7000
Enrolment FAI